In [1]:
import numpy as np

class OctreeNode:
    def __init__(self, bounds, data):
        self.bounds = bounds  # (xmin,xmax, ymin,ymax, zmin,zmax)
        self.data = data
        self.children = []
        self.is_leaf = True
        self.value = None  # 0,1 hoặc density

In [2]:
grid = np.load('Data/building_case_0.npy')
wind = np.load('Data/mean_case_0.npy')[:, :, :, [2,1,0]]

In [3]:
def is_homogeneous(block):
    return np.all(block == 0) or np.all(block == 1)

In [4]:
def split_node(node):
    x0,x1,y0,y1,z0,z1 = node.bounds
    
    xm = (x0 + x1)//2
    ym = (y0 + y1)//2
    zm = (z0 + z1)//2
    
    sub_bounds = [
        (x0,xm,y0,ym,z0,zm),
        (xm,x1,y0,ym,z0,zm),
        (x0,xm,ym,y1,z0,zm),
        (xm,x1,ym,y1,z0,zm),
        (x0,xm,y0,ym,zm,z1),
        (xm,x1,y0,ym,zm,z1),
        (x0,xm,ym,y1,zm,z1),
        (xm,x1,ym,y1,zm,z1),
    ]
    
    children = []
    for b in sub_bounds:
        bx0,bx1,by0,by1,bz0,bz1 = b
        sub_data = node.data[bx0:bx1, by0:by1, bz0:bz1]
        child = OctreeNode(b, sub_data)
        children.append(child)
        
    return children

In [5]:
def build_octree(grid, max_voxel=125):
    root = OctreeNode((0,grid.shape[0],
                       0,grid.shape[1],
                       0,grid.shape[2]), grid)
    
    leaves = [root]
    
    while True:
        # dừng nếu đủ số voxel
        if len(leaves) >= max_voxel:
            break
        
        # chọn node "phức tạp nhất"
        scores = []
        for node in leaves:
            block = node.data
            score = block.std()  # variance
            scores.append(score)
        
        idx = np.argmax(scores)
        node = leaves[idx]
        
        # nếu homogeneous thì không split
        if is_homogeneous(node.data):
            break
        
        # split
        children = split_node(node)
        
        leaves.pop(idx)
        leaves.extend(children)
    
    return leaves

In [6]:
def extract_voxels(nodes):
    points = []
    sizes = []
    values = []
    
    for n in nodes:
        x0,x1,y0,y1,z0,z1 = n.bounds
        
        cx = (x0 + x1)/2
        cy = (y0 + y1)/2
        cz = (z0 + z1)/2
        
        size = max(x1-x0, y1-y0, z1-z0)
        value = n.data.mean()
        
        points.append([cx,cy,cz])
        sizes.append(size)
        values.append(value)
    
    return np.array(points), np.array(sizes), np.array(values)

In [7]:
import open3d as o3d

def visualize(points):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    o3d.visualization.draw_geometries([pcd])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [8]:
nodes = build_octree(grid, max_voxel=125)

points, sizes, values = extract_voxels(nodes)

print("Number of voxels:", len(points))

visualize(points)

c:\Users\KJD1912\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\KJD1912\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\KJD1912\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\KJD1912\AppData\Local\Temp\ipykernel_1148\1412840835.py:14: RuntimeWarning: Mean of empty slice
  value = n.data.mean()
c:\Users\KJD1912\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Number of voxels: 22


In [ ]:
import numpy as np
import open3d as o3d

# --- Octree Node ---
class OctreeNode:
    def __init__(self, bounds, data):
        self.bounds = bounds  # (xmin,xmax, ymin,ymax, zmin,zmax)
        self.data = data
        self.children = []
        self.is_leaf = True
        self.value = None  # mean value

# --- Check if block is homogeneous ---
def is_homogeneous(block):
    return np.all(block == 0) or np.all(block == 1)

# --- Split node into 8 children ---
def split_node(node):
    x0,x1,y0,y1,z0,z1 = node.bounds
    xm = (x0 + x1)//2
    ym = (y0 + y1)//2
    zm = (z0 + z1)//2
    
    sub_bounds = [
        (x0,xm,y0,ym,z0,zm),
        (xm,x1,y0,ym,z0,zm),
        (x0,xm,ym,y1,z0,zm),
        (xm,x1,ym,y1,z0,zm),
        (x0,xm,y0,ym,zm,z1),
        (xm,x1,y0,ym,zm,z1),
        (x0,xm,ym,y1,zm,z1),
        (xm,x1,ym,y1,zm,z1),
    ]
    
    children = []
    for b in sub_bounds:
        bx0,bx1,by0,by1,bz0,bz1 = b
        sub_data = node.data[bx0:bx1, by0:by1, bz0:bz1]
        child = OctreeNode(b, sub_data)
        children.append(child)
    return children

# --- Build Octree with max_voxel constraint ---
def build_octree(grid, max_voxel=125):
    root = OctreeNode((0,grid.shape[0],
                       0,grid.shape[1],
                       0,grid.shape[2]), grid)
    leaves = [root]
    
    while True:
        if len(leaves) >= max_voxel:
            break
        # chọn node "phức tạp nhất"
        scores = [n.data.std() for n in leaves]
        idx = np.argmax(scores)
        node = leaves[idx]
        if is_homogeneous(node.data):
            # không split nữa
            leaves.pop(idx)
            leaves.append(node)
            continue
        children = split_node(node)
        leaves.pop(idx)
        leaves.extend(children)
    return leaves

# --- Extract voxel centers + size ---
def extract_voxels(nodes):
    voxels = []
    for n in nodes:
        x0,x1,y0,y1,z0,z1 = n.bounds
        size = max(x1-x0, y1-y0, z1-z0)
        center = [(x0+x1)/2, (y0+y1)/2, (z0+z1)/2]
        voxels.append((center, size))
    return voxels

# --- Visualize voxels as cube ---
def visualize_voxels(voxels):
    geometries = []
    for center, size in voxels:
        cube = o3d.geometry.TriangleMesh.create_box(width=size,
                                                    height=size,
                                                    depth=size)
        cube.translate(np.array(center) - size/2)
        cube.paint_uniform_color([0.7, 0.3, 0.3])
        geometries.append(cube)
    o3d.visualization.draw_geometries(geometries)

# --------------------------
# --- Run example ---
# grid giả lập (dùng nhỏ cho demo)
nx, ny, nz = 60, 40, 60
# grid = np.zeros((nx, ny, nz), dtype=np.uint8)
# grid[20:40, 10:30, 25:45] = 1  # obstacle block

nodes = build_octree(grid, max_voxel=125)
voxels = extract_voxels(nodes)
visualize_voxels(voxels)